# Impact of External Ressource on Detection Performance

In [1]:
import sys
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import combinations

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR, EXP_COLS, CLASS_COLS
from utils.analysis_helpers import wrap_model_comparison, wrap_model_comparison_union, print_model_comparison_results, combine_model_comparison_outputs, extract_metric_values, compute_balanced_metrics


In [2]:
PROVIDER="claude"

In [3]:
df_b_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_1.feather"))
df_d_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_1.feather"))
df_b_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_0.feather"))
df_d_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_0.feather"))

In [4]:
df_d_1.columns

Index(['comment_cleaned', 'id', 'discourse', 'comment_level',
       'comment_codes_all', 'source_outlet', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'comment_codes_all_list', 'comment_codes_all_chapters',
       'comment_codes_all_sections', 'explanation_ihra_explanation_sections',
       'explanation_tax_chapters', 'explanation_tax_ex_chapters'],
      dtype='object')

In [5]:
df_b_1.columns

Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_tax', 'explanation_tax', 'classification_tax_ex',
       'explanation_tax_ex', 'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'explanation_ihra_explanation_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters'],
      dtype='object')

In [6]:
cols_d = ['comment_cleaned', 'discourse', 'comment_level', 
          'comment_codes_all', 'source_outlet'] + EXP_COLS + CLASS_COLS

for c in cols_d: 
    if c not in df_d_0.columns:
        print(c)

In [7]:
cols_b = ['comment_cleaned', 'keyword', 'ihra_section_1', 'ihra_section_2'] + EXP_COLS + CLASS_COLS

for c in cols_b: 
    if c not in df_b_0.columns:
        print(c)

In [8]:
N_neg_total_d = 39424
N_neg_total_b = 9358

### Merge positive and negative samples

In [9]:
df_d_0["label"] = 0
df_d_1["label"] = 1
df_b_0["label"] = 0
df_b_1["label"] = 1

cols_b.insert(0,"label")
cols_d.insert(0,"label")

In [10]:
df_b = pd.concat([df_b_0[cols_b], df_b_1[cols_b]], ignore_index=True)
df_d = pd.concat([df_d_0[cols_d], df_d_1[cols_d]], ignore_index=True)

In [11]:
df_b.label.value_counts()

label
1    1858
0    1000
Name: count, dtype: int64

In [12]:
df_d.label.value_counts()

label
1    2977
0     992
Name: count, dtype: int64

### Combine datasets

In [13]:
w_neg_d = N_neg_total_d / len(df_d_0)
w_neg_b = N_neg_total_b / len(df_b_0)

df_d["dataset_id"] = "d"
df_b["dataset_id"] = "b"

df_union = pd.concat([df_d, df_b], ignore_index=True)

N_neg_total_by_dataset = {
    "d": N_neg_total_d,
    "b": N_neg_total_b,
}

### Distribution of responses in the positive samples

In [14]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.004844,NaN,NaN,NaN
No,0.414424,0.278794,0.156082,0.259957
Unsure,0.030140,0.045210,0.056512,0.032831
Yes,0.550592,0.675996,0.787406,0.707212


In [15]:
# Decoding  
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.013436,NaN,NaN,NaN
No,0.619416,0.457843,0.269735,0.422573
Unsure,0.040645,0.064830,0.056097,0.034263
Yes,0.326503,0.477326,0.674169,0.543164


### Distribution of responses in the negative samples

In [16]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.011,NaN,NaN,NaN
No,0.920,0.861,0.838,0.876
Unsure,0.011,0.046,0.038,0.023
Yes,0.058,0.093,0.124,0.101


In [17]:
# Decoding
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.040323,NaN,NaN,NaN
No,0.943548,0.889113,0.890121,0.923387
None,0.001008,NaN,NaN,NaN
Unsure,0.003024,0.089718,0.066532,0.040323
Yes,0.012097,0.021169,0.043347,0.036290


### Compute metrics and CIs on weighted union of both datasets

- B and D have the same weight (0.5)
- CI for recall reflects sampling from population; Ci for precision and F1 reflects sampling from population and 1000er sampling from dataset, thus larger CIs

In [21]:
model_comparison_metrics = compute_balanced_metrics(df_B=df_b, df_D=df_d, N_neg_total_B=N_neg_total_b, N_neg_total_D=N_neg_total_d)
model_comparison_metrics

,model,metric,value,ci_low,ci_high
0,no_kb,precision,0.662104,0.599495,0.737558
1,no_kb,recall,0.438548,0.424441,0.452904
2,no_kb,f1,0.518410,0.496796,0.540084
3,ihra,precision,0.610346,0.559160,0.671554
4,ihra,recall,0.576661,0.562677,0.590607
5,ihra,f1,0.586806,0.562325,0.613092
6,tax,precision,0.548893,0.508042,0.597075
7,tax,recall,0.730787,0.718278,0.743421
8,tax,f1,0.626331,0.598483,0.656493
9,tax_ex,precision,0.556097,0.512593,0.609207


In [22]:
model_comparison_metrics.to_parquet(f"{PROVIDER}_model_comparison_metrics.parquet", index=False)

### Export errors for qualitative analysis

In [18]:
df_b[(df_b["label"]==0) & (df_b["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_decoding.csv"), index=False)
df_b[(df_b["label"]==0) & (df_b["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_decoding.csv"), index=False)

### Statistical model comparison vs baseline (no_kb)

## Bloomington

Mapping "Yes" to 1 and Unsure to 0, class balance

In [19]:
res_b_weighted = wrap_model_comparison(df=df_b, N_neg_total=N_neg_total_b)
print_model_comparison_results(res_b_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.5907
metric(no_kb): 0.6534
Diff (model - baseline): -0.0627
95% CI: [-0.1106, -0.0178]
p-value (raw): 0.0071
p-value (adjusted): 0.0071
Effect size h: -0.1293 (95% CI: [-0.2309, -0.0367])

Model: tax
metric(tax): 0.5577
metric(no_kb): 0.6534
Diff (model - baseline): -0.0957
95% CI: [-0.1430, -0.0520]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.1961 (95% CI: [-0.2985, -0.1057])

Model: tax_ex
metric(tax_ex): 0.5816
metric(no_kb): 0.6534
Diff (model - baseline): -0.0717
95% CI: [-0.1176, -0.0285]
p-value (raw): 0.0014
p-value (adjusted): 0.0028
Effect size h: -0.1477 (95% CI: [-0.2451, -0.0583])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.6760
metric(no_kb): 0.5506
Diff (model - baseline): 0.1254
95% CI: [nan, nan]
p-value (raw): 6.106e-51
p-value (adjusted): 6.106e-51
Effect size h: 0.2583 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.7874
metric(no_kb): 0.5506
Diff (model - baseline): 0.2368
95% C

### Decoding

Mapping "Yes" to 1 and "Unsure" to 0, class balance

In [20]:
res_d_weighted = wrap_model_comparison(df=df_d, N_neg_total=N_neg_total_d)
print_model_comparison_results(res_d_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.6300
metric(no_kb): 0.6709
Diff (model - baseline): -0.0409
95% CI: [-0.1497, 0.0523]
p-value (raw): 0.4436
p-value (adjusted): 0.4436
Effect size h: -0.0857 (95% CI: [-0.3285, 0.1090])

Model: tax
metric(tax): 0.5401
metric(no_kb): 0.6709
Diff (model - baseline): -0.1307
95% CI: [-0.2557, -0.0277]
p-value (raw): 0.0135
p-value (adjusted): 0.0405
Effect size h: -0.2684 (95% CI: [-0.5540, -0.0560])

Model: tax_ex
metric(tax_ex): 0.5306
metric(no_kb): 0.6709
Diff (model - baseline): -0.1403
95% CI: [-0.2667, -0.0345]
p-value (raw): 0.0141
p-value (adjusted): 0.0405
Effect size h: -0.2876 (95% CI: [-0.5746, -0.0704])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.4773
metric(no_kb): 0.3265
Diff (model - baseline): 0.1508
95% CI: [nan, nan]
p-value (raw): 3.121e-87
p-value (adjusted): 3.121e-87
Effect size h: 0.3090 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.6742
metric(no_kb): 0.3265
Diff (model - baseline): 0.34

### Combined datasets

In [23]:
# Union of both datasets is treated according to the class distribution in the original datasets and according to the overall dataset size, thus decoding matters more
res_union_weighted = wrap_model_comparison_union(df_union, N_neg_total_by_dataset)
print_model_comparison_results(res_union_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.6109
metric(no_kb): 0.6618
Diff (model - baseline): -0.0508
95% CI: [-0.1391, 0.0356]
p-value (raw): 0.2378
p-value (adjusted): 0.2378
Effect size h: -0.1057 (95% CI: [-0.2922, 0.0738])

Model: tax
metric(tax): 0.5474
metric(no_kb): 0.6618
Diff (model - baseline): -0.1144
95% CI: [-0.1972, -0.0338]
p-value (raw): 0.0038
p-value (adjusted): 0.0114
Effect size h: -0.2346 (95% CI: [-0.4100, -0.0686])

Model: tax_ex
metric(tax_ex): 0.5523
metric(no_kb): 0.6618
Diff (model - baseline): -0.1095
95% CI: [-0.1948, -0.0251]
p-value (raw): 0.0099
p-value (adjusted): 0.0198
Effect size h: -0.2247 (95% CI: [-0.4048, -0.0511])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5537
metric(no_kb): 0.4126
Diff (model - baseline): 0.1411
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: 0.2832 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.7177
metric(no_kb): 0.4126
Diff (model - baseline): 0.3051
95% CI: [nan,

In [24]:
# Union of both datasets is treated according to the class distribution in the original datasets but both datasets weightes equally

res_union_dataset_balanced = combine_model_comparison_outputs(
    {
        "B": res_b_weighted,
        "D": res_d_weighted,
    },
    dataset_weights={
        "B": 0.5,
        "D": 0.5,
    },
)


In [25]:
print_model_comparison_results(res_union_dataset_balanced)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.6103
metric(no_kb): 0.6621
Diff (model - baseline): -0.0518
95% CI: [nan, nan]
p-value (raw): 0.6178
p-value (adjusted): 0.6178
Effect size h: -0.1075 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.5489
metric(no_kb): 0.6621
Diff (model - baseline): -0.1132
95% CI: [nan, nan]
p-value (raw): 0.0516
p-value (adjusted): 0.1548
Effect size h: -0.2323 (95% CI: [nan, nan])

Model: tax_ex
metric(tax_ex): 0.5561
metric(no_kb): 0.6621
Diff (model - baseline): -0.1060
95% CI: [nan, nan]
p-value (raw): 0.0553
p-value (adjusted): 0.1548
Effect size h: -0.2177 (95% CI: [nan, nan])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5767
metric(no_kb): 0.4385
Diff (model - baseline): 0.1381
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: 0.2837 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.7308
metric(no_kb): 0.4385
Diff (model - baseline): 0.2922
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
E

In [26]:
res_union_dataset_balanced

[('precision',
  {'ihra': {'metric_A': 0.6103460574011657,
    'metric_B': 0.6621037123686844,
    'diff': -0.05175765496751872,
    'effect_size_h': np.float64(-0.10752452555282199),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.6178),
    'p_value_adj': np.float64(0.6178)},
   'tax': {'metric_A': 0.548892946657485,
    'metric_B': 0.6621037123686844,
    'diff': -0.11321076571119931,
    'effect_size_h': np.float64(-0.23227470626236157),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.0516),
    'p_value_adj': np.float64(0.1548)},
   'tax_ex': {'metric_A': 0.556097486574455,
    'metric_B': 0.6621037123686844,
    'diff': -0.10600622579422925,
    'effect_size_h': np.float64(-0.21765130194281834),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_valu

In [27]:
df_B = extract_metric_values(res_b_weighted, "B")
df_D = extract_metric_values(res_d_weighted, "D")
df_union = extract_metric_values(res_union_weighted, "union")
df_union_bal = extract_metric_values(res_union_dataset_balanced, "union_dataset_balanced")

df_metrics = pd.concat(
    [df_B, df_D, df_union, df_union_bal],
    ignore_index=True,
)

In [28]:
df_metrics

,setting,metric,model,value
0,B,precision,no_kb,0.653355
1,B,precision,ihra,0.590699
2,B,precision,tax,0.557675
3,B,precision,tax_ex,0.581633
4,B,recall,no_kb,0.550592
5,B,recall,ihra,0.675996
6,B,recall,tax,0.787406
7,B,recall,tax_ex,0.707212
8,B,f1,no_kb,0.597588
9,B,f1,ihra,0.630476


##  Prediction overlap between models

In [29]:
def intersection_over_union(y_preds, label=1):
    y_preds = [np.asarray(y) for y in y_preds]
    intersection = np.logical_and.reduce([y == label for y in y_preds])
    union = np.logical_or.reduce([y == label for y in y_preds])
    if np.sum(union) == 0:
        return np.nan  # avoid division by zero
    return np.round(np.sum(intersection) / np.sum(union), 2)

### Bloomington

In [30]:
# all 4
intersection_over_union([df_b[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.35)

In [31]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.4
IoU for classification_no_kb_cleaned vs classification_tax: 0.44
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.42
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.81
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.85
IoU for classification_tax vs classification_tax_ex: 0.85


#### Positive samples only

In [32]:
intersection_over_union([
    df_b[df_b.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.39)

In [33]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[df_b.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[df_b.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.44
IoU for classification_no_kb_cleaned vs classification_tax: 0.48
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.45
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.83
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.87
IoU for classification_tax vs classification_tax_ex: 0.86


### Decoding

In [34]:
# all 4
intersection_over_union([df_d[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.17)

In [35]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.23
IoU for classification_no_kb_cleaned vs classification_tax: 0.28
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.25
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.68
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.76
IoU for classification_tax vs classification_tax_ex: 0.74


#### Positive samples only

In [36]:
# all 5
intersection_over_union([df_d[df_d.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.17)

In [37]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[df_d.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[df_d.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.23
IoU for classification_no_kb_cleaned vs classification_tax: 0.28
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.26
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.69
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.76
IoU for classification_tax vs classification_tax_ex: 0.75
